df

In [5]:
import re
import os
import client_lib

def modify_script(
    script_path, 
    group_index, 
    n_prompts=1100, 
    output_dir="cloud_inference", 
    # data_type='new',
    master_port=1900,
    
):
    """
    Модифицирует скрипт, добавляя только i-тую группу LORAS (по 12 штук)
    
    :param script_path: путь к исходному sh-скрипту
    :param group_index: индекс группы (начинается с 0)
    :param loras_per_group: количество LORAS в группе
    :param output_dir: директория для сохранения модифицированных скриптов
    :return: путь к новому скрипту
    """
    # Создаем директорию для результатов, если ее нет
    os.makedirs(output_dir, exist_ok=True)
    
    # Читаем исходный скрипт
    with open(script_path, 'r') as f:
        script_content = f.read()
    
    start_prompt_i = n_prompts * group_index
    
    # Модифицируем скрипт
    new_script = []
    for line in script_content.split('\n'):
        if line.strip().startswith('--start_prompt_i'):
            line = line.replace(f"{line.strip()}", f"--start_prompt_i={start_prompt_i} \\",)
        elif line.strip().startswith('--n_prompts'):
            line = line.replace(f"{line.strip()}", f"--n_prompts={n_prompts} \\",)
        elif 'master_port' in line:
            line = line.replace("--master_port=1224", "") #f"--master_port={master_port + group_index}")

        new_script.append(line)
    
    # Сохраняем новый скрипт
    output_path = os.path.join(output_dir, f"script_group_{group_index}_nocfg.sh")
    with open(output_path, 'w') as f:
        f.write('\n'.join(new_script))
    
    return output_path

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [6]:
job_idx = 1
cur_script = modify_script(
    "/home/jovyan/dmitrienko/workspace/diffusion-pipe_dmitrienko/Wan2_1/wan_nocfg_480.sh",
    job_idx, 
    n_prompts=1200*1, 
    output_dir="/home/jovyan/dmitrienko/workspace/diffusion-pipe_dmitrienko/cloud_inference",
)

In [7]:
cur_script

'/home/jovyan/dmitrienko/workspace/diffusion-pipe_dmitrienko/cloud_inference/script_group_1_nocfg.sh'

In [8]:
kandinsky_run = client_lib.Job(
    script=f"bash {cur_script}",
    n_workers=1,
    processes_per_worker=1,
    job_desc=f"dmitrienko-inference-wan-nocfg-{job_idx} \ #kandinsky.video #prod",
    #base_image='cr.ai.cloud.ru/aicloud-base-images/py3.11-torch2.4.0:0.0.40',
    base_image="cr.ai.cloud.ru/76cf88ab-4bcd-476e-806b-c956ba601728/job-kandinsky-5-train:py312-cuda12.8.1-torch2.7.rc-fa2-1.1.1",
    #conda_env="kandinsky-cuda12.8",

    instance_type='a100plus.8gpu.80vG.96C.1456G', #'a100plus.1gpu.80vG.12C.182G',
    type='binary_exp', 
    preflight_check=True, 
    region='SR008',
    priority_class='high',

)
kandinsky_run.submit()

'Job "lm-mpi-job-3aeb1332-111b-4470-821c-38ca009d4a39" created.'

In [32]:
import os

folder_path = '/home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575'  # текущая директория (или укажите свой путь)

for i in range(35):  # i от 0 до 34
    file_name = f"{i}.mp4"
    file_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Удалён файл: {file_path}")
    else:
        print(f"Файл не найден: {file_path}")

Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/0.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/1.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/2.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/3.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/4.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/5.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/6.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed5575/7.mp4
Удалён файл: /home/jovyan/dmitrienko/workspace/outputs/WAN/original/t2v_50steps_81frames_5gs_5sh_seed557